# LexFront — large-scale test on Sentiment140 (1.6 M tweets)

**What this does.** Runs the tokenizer comparison on a very large noisy dataset: whitespace, NLTK, TweetTokenizer and LexFront
with TF-IDF + Naive Bayes / Logistic Regression, on a user-grouped split (no author in both train and test), at training sizes
10 000, 100 000, 1 000 000 and the full set, plus a test with artificially noisy test text. The protocol and the hypotheses were
written down before the run: `docs/LARGE_PROTOCOL.md`. Every number in the report is read from a JSON file written by this run.

**How to run it.** CPU runtime is enough. Run all cells, once. The real run takes **hours**. Every finished fit is saved, so if
Colab disconnects, reopen this notebook and run all cells again: finished fits are skipped. With `USE_DRIVE = True` they are saved on
your Google Drive (you will be asked to allow access once), otherwise only in the runtime (lost if the runtime is deleted).

**First time?** Set `QUICK_MODE = True` in the next cell for a dry run of a few minutes on a small subsample.
At the end the cells show the report, make a zip you can download, and can optionally push the results to a new branch of the repo.

In [ ]:
# 1. Get the code, install dependencies (requirements.txt: version ranges), download NLTK data.
import os, subprocess, sys
REPO_URL = os.environ.get("LEXFRONT_REPO_URL", "https://github.com/PavitranVC/lexfront-dfa-tokenizer.git")   # env: CI/tests
REPO_DIR = "lexfront-dfa-tokenizer"

def find_repo_root():
    """The repo root at or above the working directory, or None."""
    here = os.getcwd()
    while here != os.path.dirname(here):
        if os.path.exists(os.path.join(here, "requirements.txt")) and os.path.exists(os.path.join(here, "lexer", "token_spec.py")):
            return here
        here = os.path.dirname(here)

def update_checkout():
    """Fast-forward the clone to origin. A finished run rewrites tracked files (results/quick/...) and that blocks a plain
    pull, so the local changes are set aside with `git stash` (kept: `git stash list`) and the pull is repeated."""
    def git(*args):
        return subprocess.run(["git", "-c", "user.name=colab", "-c", "user.email=colab@localhost", *args],
                              capture_output=True, text=True)
    pull = git("pull", "-q", "--ff-only")
    if pull.returncode != 0 and git("stash", "push", "--include-untracked", "-q").returncode == 0:
        print("local changes from an earlier run were set aside (git stash list) so the code could be updated")
        pull = git("pull", "-q", "--ff-only")
    if pull.returncode != 0:
        raise RuntimeError("could not update the existing clone, so it would run old code under this newer notebook.\n"
                           f"Delete the folder (!rm -rf {os.getcwd()}) and run this cell again.\n" + pull.stderr.strip())

root = find_repo_root()
if root is not None:                               # running inside a checkout (local, CI)
    os.chdir(root)
elif os.path.isdir(REPO_DIR):                      # Colab after a restart or a new session: already cloned
    os.chdir(REPO_DIR)
    update_checkout()                              # old code under this newer notebook would fail further down
else:                                              # fresh Colab runtime: clone the repo
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_DIR)
if not os.path.exists(os.path.join("lexer", "token_spec.py")):
    raise RuntimeError(f"{os.getcwd()} is not a complete checkout: delete it and run this cell again")
print("working directory:", os.getcwd())
sys.path.insert(0, os.getcwd())
for name in [m for m in sys.modules if m.split(".")[0] in ("experiments", "lexer", "subword")]:
    del sys.modules[name]                          # re-run in a live kernel: do not keep the old code imported

from experiments import setup_check               # standard library only
before_install = setup_check.snapshot()
%pip install -q -r requirements.txt
setup_check.report_after_install(before_install)  # prints versions in use; warns if a restart is needed
import nltk
for package in ["punkt_tab"]:                      # what word_tokenize needs; a failed download must not pass silently
    try:
        nltk.data.find(f"tokenizers/{package}")
    except LookupError:
        nltk.download(package, quiet=True)
        try:
            nltk.data.find(f"tokenizers/{package}")
        except LookupError:
            # Still missing. With a proxy configured NLTK refuses to download (its SSRF guard, CWE-918: the proxy, not
            # NLTK, picks the destination); it detects proxies through urllib (any *_proxy variable, the system settings),
            # so instead of guessing which one is set, retry once with the opt-in, for this one download only.
            import nltk.pathsec as pathsec
            before = os.environ.get("NLTK_ALLOW_PROXIED_URLOPEN"), getattr(pathsec, "ALLOW_PROXIED_FETCH", None)
            print("download of", package, "failed or was refused; retrying once with NLTK's proxy opt-in")
            os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
            pathsec.ALLOW_PROXIED_FETCH = True
            try:
                nltk.download(package, quiet=True)
            finally:
                pathsec.ALLOW_PROXIED_FETCH = before[1]
                if before[0] is None:
                    os.environ.pop("NLTK_ALLOW_PROXIED_URLOPEN", None)
                else:
                    os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = before[0]
        nltk.data.find(f"tokenizers/{package}")    # raises LookupError with the download hint if it is still missing

In [ ]:
import os
QUICK_MODE = os.environ.get("LEXFRONT_LARGE_QUICK", "0") == "1"   # True: dry run on a small subsample (minutes). False: the real run (hours).
USE_DRIVE = True        # keep the per-run files on Google Drive so a disconnected session resumes (ignored when not on Colab)
PUSH_RESULTS = False    # True: at the end push results to a NEW branch of the repo (needs a Colab secret GITHUB_TOKEN, see the last cell)

WORK = "../lexfront_large_work_quick" if QUICK_MODE else "../lexfront_large_work"
OUT = "results/quick/large" if QUICK_MODE else "results/large"
if USE_DRIVE and not QUICK_MODE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/lexfront_large_work"
    except Exception as error:      # not on Colab (ImportError) or the Drive permission was declined
        print("Google Drive not used (" + type(error).__name__ + "): the per-run files stay in the runtime, next to the repository")
print("QUICK_MODE =", QUICK_MODE, "| per-run files:", WORK, "| results:", OUT)

## Run

Loads the dataset, splits by user, cleans (`experiments/clean.py`), audits it and runs every fit. Progress lines say `done` for a fit
that was just computed and `cached` for one that was found from an earlier session.

In [ ]:
from experiments import run_large
run_large.main(["--work", WORK] + (["--quick"] if QUICK_MODE else []))

## Report

Generated from the JSON files; includes the pre-registered hypotheses with what was observed.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/REPORT.md").read()))

## Save the results

A zip of the results folder. On Colab it is downloaded to your computer; copy the files into the repository's `results/large/`.

In [ ]:
zip_path = run_large.make_zip(OUT, "lexfront_large_results.zip")
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass

## Optional: push the results to a new branch

Only if `PUSH_RESULTS = True`. It commits the results folder to a **new branch** (never `main`) and pushes it, so the results can be reviewed
and merged. It needs a GitHub token stored as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar, switch on "Notebook access").
Use a fine-grained token limited to this one repository with *Contents: read and write*. The token is handed to git through an
askpass helper and is never printed.

In [ ]:
if PUSH_RESULTS:
    from google.colab import userdata
    from datetime import datetime, timezone
    branch = "results/large-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
    print(run_large.push_results(".", OUT, branch, userdata.get("GITHUB_TOKEN")))
else:
    print("PUSH_RESULTS is False: nothing pushed")